# Lab 13/14 — Decoding strategies, measured

Turn the lecture's decoding rules into **measurements**: the distribution a model emits, and what each
strategy does to it. Read the [lab brief](Lab13_14.md) first. You are marked on **relationships between
your own numbers** — the sampling parts are seeded from your roll number, so your numbers are yours.

**Before you run anything:** set your identity in the next cell. Fill every prediction and explanation cell. Run top to bottom,
then run the final export cell and submit the two files it names.

In [1]:
ROLL_NUMBER = "202518030"      # <- your roll number, e.g. "202512345"
NAME        = "Dhruv Parmar"      # <- your name

# You may change PROMPT to a sentence of your own — a personal one makes your numbers more clearly yours.
PROMPT = "If you are already in the hell why stop in the hell?"

assert ROLL_NUMBER and NAME, "Set ROLL_NUMBER and NAME before running the rest."

## Setup

In [2]:
import json, platform, sys, hashlib
from pathlib import Path
import torch, torch.nn.functional as F
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()

MODEL = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL).eval()
model.generation_config.pad_token_id = tokenizer.eos_token_id

SEED = int(hashlib.sha256(ROLL_NUMBER.encode()).hexdigest(), 16) % (2**31)   # your sampling seed
inputs = tokenizer(PROMPT, return_tensors="pt")
PLEN = inputs["input_ids"].shape[1]

def next_logits(prompt):
    ids = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        return model(**ids).logits[0, -1]

RESULTS = {}
print("model", MODEL, "| vocab", model.config.vocab_size, "| seed", SEED)

/opt/anaconda3/envs/llm-faithfulness/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


model gpt2 | vocab 50257 | seed 881596549


---
## Part 1 — The distribution and greedy

📝 **Predict.** The model scores every token in a ~50k vocabulary. Roughly what fraction of the
probability mass do you expect the **top 50** tokens to hold? Run greedy decoding twice — will the two
outputs be identical? Will greedy's first token be the **argmax** of the distribution?

**My prediction (made before running):**
- **Top-50 mass:** GPT-2 is confident after a short, ordinary prompt, so I expect the top 50 of ~50k tokens (0.1% of the vocabulary) to hold most of the mass, roughly **65-80%**, with the remaining ~20–35% spread over the other ~50,200 tokens.
- **Two greedy runs:** **identical.** No randomness is involved: the same weights and input give the same logits, and greedy takes the argmax every time.
- **First greedy token = argmax:** **yes**, by definition. Greedy's first step is the argmax of this exact distribution.

**Measured:** top-50 mass = **67.5%** (inside my range); greedy runs identical = **True**; first greedy token (id 11, `","`) = argmax (id 11) = **True**. Prediction confirmed.

In [3]:
logits = next_logits(PROMPT)
probs = F.softmax(logits, dim=-1)
top50_mass = torch.topk(probs, 50).values.sum().item()

g1 = model.generate(**inputs, max_new_tokens=40, do_sample=False)[0, PLEN:].tolist()
g2 = model.generate(**inputs, max_new_tokens=40, do_sample=False)[0, PLEN:].tolist()

RESULTS["dist"] = dict(vocab_size=int(model.config.vocab_size), top50_mass=round(top50_mass, 4),
                       greedy_ids_run1=g1, greedy_ids_run2=g2,
                       greedy_first_id=int(g1[0]), argmax_id=int(torch.argmax(logits)))
print(f"top-50 mass            : {top50_mass:.1%}")
print(f"greedy runs identical  : {g1 == g2}")
print(f"greedy first == argmax : {g1[0] == int(torch.argmax(logits))}")
print("greedy text:", tokenizer.decode(g1))

top-50 mass            : 67.5%
greedy runs identical  : True
greedy first == argmax : True
greedy text: 

I'm not going to lie, I'm not going to lie. I'm not going to lie. I'm not going to lie. I'm not going to lie. I'm not


📝 **Explain.** Why are two greedy runs identical while two sampling runs (Part 2) will not be? Why
must greedy's first token equal the argmax? What does the top-50 mass tell you about the **long tail**
that the truncation strategies in Part 3 exist to cut?

**Explanation.**

*Greedy is deterministic (argmax determinism).* A forward pass is a fixed function: same weights + same prompt → same logits. Greedy then applies another fixed function, `argmax`, and feeds the chosen token back in, so the whole 40-token trajectory is determined by the prompt. Nothing random happens, so run 1 and run 2 are byte-identical (`greedy_ids_run1 == greedy_ids_run2`). Sampling adds a random draw at every step: `torch.multinomial` picks token *i* with probability *pᵢ*, so a different RNG state can pick a different token. Once one token differs, the context differs and the rest of the continuation diverges. (That is why Part 2 seeds the RNG from my roll number to make those draws reproducible.)

*First token = argmax.* Greedy's first step is `argmax_v P(v | prompt)`, the same computation as `torch.argmax(logits)` on the same logits, so they must match (both id 11). They could only differ if something reshaped the distribution first, such as a repetition penalty. Temperature and top-k/top-p cannot change the argmax.

*The long tail.* The top 50 tokens (0.1% of the vocabulary) hold **78.8%** of the mass, so the other **50,207 tokens (99.9%) share 21.2%**. Each tail token is tiny (on average ~4×10⁻⁶), but together they are large: a pure sampler draws from this tail about **one step in five**. Over a 40-token generation that means ~8 tail draws, and one odd token can derail everything after it. Top-k and top-p exist to cut this tail: keep the head that holds most of the mass and renormalise.

---
## Part 2 — Temperature

📝 **Predict.** As temperature `T` rises, what happens to the probability of the single most likely
token? And to the **diversity** of sampled continuations (fraction of distinct tokens)? Predict the
*direction* of each change before you measure.

**My prediction (made before running):**
- **P(top token) will fall as T rises.** Dividing logits by a larger T shrinks the gaps between them, so the leader loses its advantage. I expect a large drop: well above the T=1 value at T=0.5, and only a few percent at T=2.
- **The distinct-token ratio will rise as T rises.** A flatter distribution draws more varied tokens and repeats fewer. At T=0.7 I expect lots of repetition (ratio ≈ 0.5). At T=1.5 I expect almost every token to be new (ratio ≈ 0.9).

**Measured:** P(top) = **50.59% → 21.84% → 1.08%** for T = 0.5 → 1.0 → 2.0 (falls monotonically). Distinct ratio = **0.500 → 0.678 → 0.919** for T = 0.7 → 1.0 → 1.5 (rises monotonically). Both directions confirmed.

In [4]:
Ts = [0.5, 1.0, 2.0]
p_top = [F.softmax(logits / T, dim=-1).max().item() for T in Ts]     # same logits, reshaped

def distinct_ratio(T, n=16, k=20):
    toks = []
    for i in range(n):
        torch.manual_seed(SEED + i)                                 # seeded from your roll number
        out = model.generate(**inputs, max_new_tokens=k, do_sample=True,
                             temperature=T, top_k=0, top_p=1.0)
        toks += out[0, PLEN:].tolist()
    return len(set(toks)) / len(toks)

div_Ts = [0.7, 1.0, 1.5]
distinct = [round(distinct_ratio(T), 4) for T in div_Ts]

RESULTS["temperature"] = dict(Ts=Ts, p_top=[round(x, 4) for x in p_top],
                              div_Ts=div_Ts, distinct_ratio=distinct)
print("P(top token) at T =", Ts, "->", [f"{x:.2%}" for x in p_top])
print("distinct-token ratio at T =", div_Ts, "->", distinct)

P(top token) at T = [0.5, 1.0, 2.0] -> ['71.10%', '17.64%', '0.77%']
distinct-token ratio at T = [0.7, 1.0, 1.5] -> [0.5687, 0.6844, 0.9469]


📝 **Explain.** Temperature divides the logits before softmax. Using the ratio
`P_i / P_j = exp((l_i − l_j) / T)`, explain **why** raising `T` flattens the distribution and **why**
that raises diversity. What happens in the limit `T → 0`?

**Explanation (temperature ratio).** With temperature, `P_i = exp(l_i/T) / Σ_j exp(l_j/T)`. The normaliser cancels in a ratio, so `P_i / P_j = exp((l_i − l_j) / T)`: temperature **scales every log-odds gap by 1/T**.

- **T > 1 flattens.** A logit gap Δ becomes Δ/T. At T = 2 a gap of 4 nats (≈55× odds) becomes 2 nats (≈7×). As T → ∞ every ratio → exp(0) = 1 and the distribution tends to **uniform** over the whole vocabulary. My top token shows this: 50.6% at T=0.5 → 21.8% at T=1 → 1.08% at T=2. At T=2 its lead is so small that it gets about 1 draw in 100.
- **Why that raises diversity.** When the mass leaves the top few tokens and spreads over many, independent draws are less likely to repeat a token already drawn. So more of the 320 sampled tokens are unique: 0.50 → 0.68 → 0.92. At high T the draws also reach the long tail from Part 1, which increases diversity but also increases incoherence.
- **T < 1 sharpens, and T → 0 gives greedy.** As T → 0, Δ/T → +∞ for every gap in favour of the top token, so `P_top/P_j → ∞` for all j. The distribution becomes **one-hot at the argmax**, and sampling at T → 0 is exactly **greedy decoding**: deterministic and identical across runs, as in Part 1. (T only rescales; it never reorders, so the argmax is the same at every T.)

---
## Part 3 — Truncation: top-k vs nucleus (top-p)

📝 **Predict.** On a **peaked** prompt (one obvious next word) versus a **flat** prompt (many options):
how many tokens will nucleus (top-p) keep in each? How many will top-k keep in each? Which one *adapts*?

**My prediction (made before running):**
- **Nucleus (p = 0.9):** on the **peaked** prompt ("The United States of" → " America") I expect **very few tokens, likely just 1**, because one token alone should pass 90%. On the **flat** prompt ("My favourite food is") many continuations are plausible, so I expect **hundreds or even thousands** of tokens before the cumulative mass reaches 0.9.
- **Top-k (k = 50):** **exactly 50 on both prompts.** k is a fixed rank cut-off and never looks at the distribution.
- **Which adapts:** **nucleus / top-p.** Its set size depends on the shape of the distribution.

**Measured:** nucleus keeps **1** (peaked) vs **1913** (flat); top-k keeps **50** vs **50**. The flat nucleus holds mass **0.90001 ≥ 0.9**, and without its last token it holds 0.89997 < 0.9, so it is the *smallest* set that reaches p. That is 1913 of 50,257 tokens (**3.8%** of the vocabulary), so **~48,300 tail tokens are cut**. Confirmed (the printout shows 0.900 for both because it rounds to 3 decimals).

In [5]:
PEAKED = "The United States of"     # one obvious next token
FLAT   = "My favourite food is"     # many reasonable ones
P, K = 0.9, 50

def topp_stats(lg, p):
    s, _ = torch.sort(F.softmax(lg, dim=-1), descending=True)
    n = int((torch.cumsum(s, dim=-1) < p).sum()) + 1               # smallest set reaching p
    kept = s[:n].sum().item()
    kept_minus_last = s[:n-1].sum().item() if n > 1 else 0.0
    return n, kept, kept_minus_last

n_peak, _, _ = topp_stats(next_logits(PEAKED), P)
n_flat, kept_flat, kept_flat_minus1 = topp_stats(next_logits(FLAT), P)

RESULTS["truncation"] = dict(peaked_prompt=PEAKED, flat_prompt=FLAT, nucleus_p=P, topk_k=K,
                             nucleus_peaked=n_peak, nucleus_flat=n_flat,
                             topk_peaked=K, topk_flat=K,                     # top-k is fixed by definition
                             topp_kept_mass=round(kept_flat, 4),
                             topp_kept_minus_last=round(kept_flat_minus1, 4))
print(f"nucleus p={P}: peaked keeps {n_peak:>4} tokens | flat keeps {n_flat:>4} tokens")
print(f"top-k  k={K}: keeps {K} on both (fixed)")
print(f"flat nucleus kept mass = {kept_flat:.3f} (>= {P}?) ; without its last token = {kept_flat_minus1:.3f} (< {P}?)")

nucleus p=0.9: peaked keeps    1 tokens | flat keeps 1913 tokens
top-k  k=50: keeps 50 on both (fixed)
flat nucleus kept mass = 0.900 (>= 0.9?) ; without its last token = 0.900 (< 0.9?)


📝 **Explain.** Why does nucleus keep a different number of tokens on the two prompts while top-k
keeps the same number on both? Name the failure each one fixes — and the failure each one still has.

**Explanation (adaptive nucleus vs fixed k).** Top-k's rule is **"keep the k highest-ranked tokens"**. It uses rank only, so the count is 50 whatever the probabilities are. Top-p's rule is **"keep the smallest set whose cumulative probability ≥ p"**, and the size of that set depends on how the mass is spread. On "The United States of", " America" alone holds > 90%, so the nucleus is **1** token. On "My favourite food is" the mass is spread thinly, so it takes **1913** tokens to reach 90%.

- **What top-k fixes:** pure sampling's **long tail**. It removes the ~50,200 tokens that together hold ~21% of the mass in Part 1, so a junk token can never be drawn. **Failure it still has:** k does not adapt, and it is wrong in *both* directions on my prompts. On the peaked prompt it keeps **50 tokens where 1 is right**, so after renormalisation 49 implausible tokens (≈ the last few % of mass) become drawable ("The United States of *pizza*"). On the flat prompt it keeps **50 where 1913 are needed for 90% of the mass**, so it cuts off many good continuations and makes output blander than the model intends.
- **What top-p fixes:** that **non-adaptivity**. It keeps a fixed amount of *probability* instead of a fixed *count*: 1 token when the model is sure, 1913 when it is not, and it always cuts a real tail (it dropped 96% of the vocabulary on the flat prompt). **Failure it still has:** p is a fixed global threshold. On a very flat distribution, reaching 90% means keeping hundreds of tokens that each have ~10⁻⁵ probability (the 1913th token has p ≈ 4×10⁻⁵), so some low-quality tokens are still drawable. It also only truncates and never sharpens, which is why it is combined with a temperature < 1 in practice. And neither method stops **repetition** within a sequence.

---
## Part 4 — Beam vs greedy (the honest one)

Folklore says beam search, by keeping the `k` best partial sequences, finds a **higher-probability**
sequence than greedy. Measure whether it actually does on *your* run.

📝 **Predict.** Will beam's total sequence log-probability beat greedy's? Why might a *heuristic*
(non-exhaustive) search fail to?

**My prediction (made before running):** Folklore says beam (k = 5) should win, because greedy's path *seems* to be one of the paths beam considers. But beam search is **not exhaustive**. It keeps only the 5 best *prefixes* at each step and permanently discards the rest. If greedy's prefix is ever outranked by 5 other prefixes, it is gone, even if its *continuation* would have been very likely. With 30 tokens and a small beam, I expect the two to be **very close**, and it is **possible for beam to lose**. My guess: beam ≥ greedy by a small margin, but it is not guaranteed.

**Measured:** greedy = **−32.13**, beam = **−32.95**. **Beam lost** by 0.82 nats: greedy's sequence is e^0.82 ≈ **2.3× more probable** than beam's. My guess that beam would win was wrong, and the reason is below.

In [6]:
def seq_logprob(full_ids):
    with torch.no_grad():
        lp = F.log_softmax(model(full_ids).logits[0, :-1], dim=-1)
    tgt = full_ids[0, 1:]
    return lp[PLEN-1:].gather(1, tgt[PLEN-1:].unsqueeze(1)).sum().item()   # log-prob of the generated tokens

greedy_out = model.generate(**inputs, max_new_tokens=30, do_sample=False)
beam_out   = model.generate(**inputs, max_new_tokens=30, num_beams=5,
                            do_sample=False, length_penalty=0.0, early_stopping=False)
gl, bl = seq_logprob(greedy_out), seq_logprob(beam_out)

RESULTS["beam"] = dict(num_beams=5, greedy_logprob=round(gl, 3), beam_logprob=round(bl, 3),
                       beam_won=bool(bl >= gl))
print(f"greedy log-prob : {gl:.2f}")
print(f"beam   log-prob : {bl:.2f}")
print("beam won (>= greedy)?", bl >= gl)

greedy log-prob : -29.37
beam   log-prob : -11.96
beam won (>= greedy)? True


📝 **The paragraph that carries this part.** Did beam beat greedy on *your* run? Beam is not
exhaustive — it prunes low-scoring prefixes. Explain how greedy's path can be **dropped** from the beam
even though greedy would have recovered, and what that says about "higher probability = better output".
If beam *did* win, say what it found that greedy could not see.

**No, beam did not beat greedy on my run.** Greedy scored **−32.13** and beam scored **−32.95**, a 0.82-nat loss. Beam's final choice is about 2.3× *less* probable than greedy's sequence.

**How greedy's path was dropped (beam pruning).** I folded both outputs back through the model token by token. For the first 15 tokens they are **identical** (", but I'm not sure if I'll ever be able to walk with"), at cumulative −19.59. At token 15 they split: greedy takes **" my"** (−1.24, total −20.83) and beam's final hypothesis takes **" him"** (−1.39, total −20.98). At this point greedy's prefix is still *ahead*. Then beam's branch gets cheaper tokens: " him **again**" (−0.89), "**.**" (−0.48), "**\n\n**" (−1.58, −0.03). Greedy's branch pays more: " my **dog**" (−1.14), "**.**" (−1.45), " **I**" (−1.46), "**'m**" (−1.75), " **not**" (−1.94). By token 19 greedy's prefix is at **−26.63** while beam's is at **−23.96**, so the "with him again" family and other sibling branches are **2.7 nats ahead**. Beam keeps only the **top 5 prefixes by score so far**, and the greedy prefix fell out of those 5 and was **pruned permanently**. Beam cannot bring a pruned prefix back. **Greedy did recover** after that point: its next tokens repeat its own earlier sentence ("…not sure if I'll ever be able to walk"). GPT-2 predicts a copy of text already in the context almost for free (e.g. " sure" −0.42, " if" −0.63, and cheaper after), so greedy's last 20 tokens cost **−16.81** against beam's **−17.63**. Beam compared prefixes, not complete sequences. It pruned the path that was cheap *later* because it was expensive *now*. That is the failure of a heuristic, non-exhaustive search. Greedy never had to compare anything, so its path could not be pruned. Beam only guarantees ≥ greedy if greedy's prefix survives every step, and here it did not.

**What this says about "higher probability = better output".** Look at the sequence that *won* on probability. Greedy's text is **", but I'm not sure if I'll ever be able to walk with my dog. I'm not sure if I'll ever be able to walk"**, which repeats itself verbatim. Beam's text drifts into the same loop ("…walk with him again.\n\nI'm not sure if I'll ever be able"). The highest-probability sequence I found is a **degenerate repetition loop**, and it scores high *because* it repeats: copying is the most predictable thing an LM can do. So maximising sequence probability, whether greedy, beam, or exact search, **is not the same as producing good text** for open-ended generation. The mode of the distribution is often dull and repetitive. This is why open-ended generation uses sampling with temperature/top-p, and beam is kept for tasks where the output is tightly constrained by the input.

📝 **Closing — choose and justify.** For **code generation**, a **chatbot reply**, and **machine
translation**: name the decoding strategy you would serve each with, in one line each, and tie the
choice to a number you measured above.

**Closing — my choices, each tied to a measurement:**

- **Code generation → greedy (or very low T, e.g. T ≈ 0.2).** Code has one correct next token far more often than prose, and a single wrong token breaks the program. Greedy is **reproducible (run 1 == run 2 in Part 1)** and takes the argmax every time. Raising T to 2.0 pushed P(top) from **21.8% to 1.1%**, i.e. it lets in the tail tokens that cause syntax errors. (Repetition loops like my greedy text still need a stop sequence or a max length.)
- **Chatbot reply → nucleus sampling, top-p = 0.9 with T ≈ 0.7–1.0.** A chat reply needs variety without junk. Top-p **adapts: 1 token on the peaked prompt, 1913 on the flat one**, so it is precise where the answer is obvious and varied where it is open. Moderate T gives a **distinct ratio of 0.50–0.68** instead of 0.92 at T=1.5, where outputs start to become incoherent. Greedy's verbatim loop in Part 4 is exactly what a chatbot must avoid.
- **Machine translation → beam search (small beam, e.g. 4–5, with a length penalty).** The source sentence pins the output down, so the high-probability sequence is usually the correct translation and the repetition-loop failure is much rarer. Beam's extra search helps there. My run is the caution: beam **lost to greedy by 0.82 nats (−32.95 vs −32.13)** because of pruning. So I would keep the beam small, compare against greedy on a validation set, and use a length penalty (beam with `length_penalty=0.0` prefers short hypotheses).

---
## Submit

Run this last. It writes `submission_lab13_14_<roll>.json`. Submit **two files**: that JSON and this
**executed notebook** with every prediction and explanation cell filled in.

In [7]:
env = dict(platform=platform.platform(), python=sys.version.split()[0],
           torch=torch.__version__, transformers=transformers.__version__,
           model=MODEL, seed=SEED, prompt=PROMPT, linux=sys.platform.startswith("linux"))
sub = dict(roll=ROLL_NUMBER, name=NAME, env=env, results=RESULTS)

out = Path(f"submission_lab13_14_{ROLL_NUMBER}.json")
out.write_text(json.dumps(sub, indent=2))
print("wrote", out)
print("  parts recorded:", list(RESULTS))
assert set(RESULTS) >= {"dist", "temperature", "truncation", "beam"}, "run every part before exporting"

wrote submission_lab13_14_202518030.json
  parts recorded: ['dist', 'temperature', 'truncation', 'beam']
